# Lesson 2 Exercises: Prompting Techniques & Structured Extraction

These exercises build on the demo notebook. You'll practice the core prompting techniques shown in class — from few-shot examples to chain-of-thought reasoning — then apply them to structured data extraction.

**Setup:** run the cell below first, then work through exercises 0–5 in order.

**Rules:**
- Write your solutions where you see `# YOUR CODE HERE`
- Read the hints if you're stuck, but try without them first
- Run each cell and inspect the output yourself — does it look right?

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
STRONG_MODEL = "gpt-5.2"  # for comparison
WEAK_MODEL = "gpt-4.1-nano"
print("Client ready")

---
## Exercise 0: Warm-Up — Structured Extraction from a Text File

Before we get into prompting techniques, let's start with the most basic LLM superpower: **turning unstructured text into structured data**.

You'll load the opening of *A Scandal in Bohemia* (the first Sherlock Holmes short story) and ask the model to extract a structured JSON summary — with no special tricks, just a clear prompt.

This exercise gets you comfortable with:
- Loading text and sending it to the API
- Writing a prompt that specifies a JSON output format
- Parsing the result with `json.loads()`

In [ ]:
# Step 1: Load the text file
# The file is the opening of "A Scandal in Bohemia" — the first Sherlock Holmes short story.

with open("prompting/sherlock-short.txt") as f:
    sherlock_text = f.read()

print(f"Loaded {len(sherlock_text)} characters")
print(f"First 200 chars:\n{sherlock_text[:200]}...")

In [ ]:
# Step 2: Write a prompt that extracts structured info from this text.
#
# Your prompt should ask the model to return a JSON object with these fields:
#   - title (string): the title of the book/story
#   - author (string): who wrote it
#   - characters (list of objects): each with "name" and "description" (1 sentence)
#   - plot_summary (string): 2-3 sentence summary of what happens in this excerpt
#   - setting (object): with "locations" (list of strings) and "time_period" (string)
#
# Requirements:
#   - Use a system message to set the task, and pass the text as the user message
#   - Tell the model to return ONLY valid JSON (no markdown, no commentary)
#   - Parse the response with json.loads()

# YOUR CODE HERE
# 1. Write your prompt (system message)
# 2. Call client.chat.completions.create(...)
# 3. Parse the response into a dict

book_info = None  # replace with your parsed dict

print(json.dumps(book_info, indent=2))

---
## Exercise 1: Schema Design — What Would You Extract?

In Exercise 0 the schema was given to you. In practice, **you design the schema** based on what's useful to extract.

Look at the Sherlock text you already loaded (`sherlock_text`). There's a lot of information in there — characters, locations, relationships, cases, time references, personality traits...

Your task:

1. **Design a JSON schema** that captures the most interesting structured information from this text. Think about: what fields? what types? what's nested vs flat?
2. **Write an extraction prompt** that includes your schema as an example JSON in the prompt itself — show the model exactly what shape you want
3. **Parse the result** with `json.loads()`

There are no wrong answers — the goal is to practice thinking about schema design *before* you write the prompt.

In [ ]:
# Design your own schema for the Sherlock text and write an extraction prompt.
#
# Steps:
#   1. Think about what's interesting to extract: characters, locations,
#      relationships, cases mentioned, personality traits, time references...
#   2. Define your schema as an example JSON object
#   3. Write a prompt that includes the example JSON so the model knows the exact shape
#   4. Send sherlock_text and parse the result
#
# Tip: embedding an example JSON directly in the prompt is one of the most
# effective ways to get consistent structured output.

# YOUR CODE HERE
# 1. Design your schema (write an example JSON in your prompt)
# 2. Call client.chat.completions.create(...)
# 3. Parse the response with json.loads()

extracted = None  # replace with your parsed dict

print(json.dumps(extracted, indent=2))

Here is the NER prompt from the demo, for reference:

In [ ]:
NER_PROMPT_TEMPLATE = """Extract all named entities from the following text and return them as JSON.

Classify each entity as one of: PERSON, ORGANIZATION, LOCATION, DATE, or MISC.

Text: "{text}"

Return only valid JSON in this format:
{{
  "entities": [
    {{"text": "...", "type": "...", "start": <char_offset>}}
  ]
}}"""


def extract_entities(text, model=MODEL, temperature=0):
    """Helper: extract NER entities from text using our template. Returns (raw_string, parsed_dict_or_None)."""
    prompt = NER_PROMPT_TEMPLATE.format(text=text)
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    raw = r.choices[0].message.content
    try:
        parsed = json.loads(raw)
        return raw, parsed
    except json.JSONDecodeError:
        return raw, None

---
## Exercise 2: Few-Shot Learning — Emotion in Watson's Narration

Watson's narration is rich with feeling — admiration, concern, nostalgia, amusement. But LLMs don't always pick up on subtle emotional tone unless you **show them what you mean**.

You'll use the Sherlock text from Exercise 0 (already loaded as `sherlock_text`) and ask the model to find passages that convey strong emotion.

You'll compare:
- **Part A:** Zero-shot — just describe the task
- **Part B:** Few-shot — include example passage/emotion pairs in your prompt

There are no "correct" answers — emotions are subjective! The point is to see how **adding examples changes the model's output**.

In [ ]:
# Part A: Zero-shot — ask the model to find emotionally rich passages and classify them.
#
# Write a prompt that:
#   - Takes the sherlock_text as input
#   - Asks the model to identify 5-6 passages that convey strong emotion
#   - For each passage, classify the dominant emotion in 1-2 words
#   - Returns JSON: [{"passage": "...", "emotion": "..."}]
#
# Use a system message for instructions, and pass sherlock_text as the user message.

# YOUR CODE HERE
# 1. Write your system prompt
# 2. Call client.chat.completions.create(...)
# 3. Parse and print the result

zero_shot_result = None  # replace with your parsed list

print("=== Part A: Zero-shot ===\n")
print(json.dumps(zero_shot_result, indent=2))

### Part B: Few-shot — teach by example

Now add **few-shot examples** to your prompt. Pick short fragments from the text — just the words around the emotion, not full sentences — and label them. Also include **negative examples** (what you do NOT want) to steer the model away from common mistakes.

For instance, your system prompt could include something like:

```
Good examples (short, focused on the emotional core):
- "alternating from week to week between cocaine and ambition" → restless compulsion
- "seized with a keen desire to see Holmes again" → fond longing
- "with a gibe and a sneer" → cold disdain

BAD examples (too long, too vague — do NOT do this):
- Do NOT quote entire paragraphs — extract only the emotionally charged fragment
- Do NOT use generic labels like "positive" or "negative" — be specific
- Do NOT include narration that describes facts without feeling
```

Then send `sherlock_text` again and compare the output to Part A.

In [ ]:
# Part B: Few-shot — add example passage/emotion pairs to your prompt.
#
# Pick 3 passages from sherlock_text and label them with 1-2 word emotions.
# Include these examples in your system prompt, then send sherlock_text again.
#
# Same output format: [{"passage": "...", "emotion": "..."}]

# YOUR CODE HERE
# 1. Write your system prompt WITH examples
# 2. Call client.chat.completions.create(...)
# 3. Parse and print the result

few_shot_result = None  # replace with your parsed list

print("=== Part B: Few-shot ===\n")
print(json.dumps(few_shot_result, indent=2))

In [ ]:
# Compare: print zero-shot vs few-shot side by side

print("=== Comparison ===\n")
print("Zero-shot emotions:")
for item in zero_shot_result:
    print(f"  {item['emotion']:20s}  <- \"{item['passage'][:65]}...\"")

print("\nFew-shot emotions:")
for item in few_shot_result:
    print(f"  {item['emotion']:20s}  <- \"{item['passage'][:65]}...\"")

print("\nLook at the differences:")
print("  - Did the model pick the same passages or different ones?")
print("  - Did the few-shot examples steer the model's emotion vocabulary?")
print("  - Are the few-shot labels more specific / nuanced?")

---
## Exercise 3: Chain-of-Thought Prompting

Some problems are easy for the model to answer directly. Others cause it to stumble — unless you ask it to **think step by step** before answering. This is **chain-of-thought (CoT)** prompting.

You'll compare:
- **Part A: Direct prompting** — just ask for the answer
- **Part B: CoT prompting** — ask the model to reason step by step, then give the answer

### Multi-phase water tank

This problem has three phases with different fill rates. Models often lose track of the state transitions.

In [ ]:
TANK_QUESTION = """A water tank holds 120 liters.
- Tap A fills at 3 liters/minute and is opened at time 0.
- Tap B fills at 5 liters/minute and is opened 10 minutes later.
- A drain empties at 4 liters/minute and is opened 5 minutes after Tap B.
How many minutes after time 0 is the tank completely full?"""

# Work it out:
#   Phase 1 (0-10 min):  3 L/min x 10 = 30 L
#   Phase 2 (10-15 min): (3+5) = 8 L/min x 5 = 40 L -> total 70 L
#   Phase 3 (15+ min):   (3+5-4) = 4 L/min, need 50 L more -> 50/4 = 12.5 min
#   Total: 15 + 12.5 = 27.5 minutes
TANK_ANSWER = 27.5

In [ ]:
# Part A: Direct prompting — ask for just the number.
#
# Run 5 times with temperature=1 to see how often it gets it right.

print(f"Question: {TANK_QUESTION}")
print(f"Correct answer: {TANK_ANSWER}\n")

print("=== Direct prompting (5 runs, temp=1) ===")
direct_correct = 0
for run in range(1, 6):
    # YOUR CODE HERE
    # 1. Call the API with a system prompt like: "Answer with just the number, nothing else."
    # 2. Extract the number from the response
    # 3. Compare to TANK_ANSWER
    answer = ""  # replace
    print(f"  Run {run}: {answer}")

print(f"\nDirect: {direct_correct}/5 correct")

In [ ]:
# Part B: CoT prompting — ask the model to think step by step BEFORE answering.
#
# Run 5 times with the same temperature=1.

print("=== Chain-of-thought prompting (5 runs, temp=1) ===")
cot_correct = 0
for run in range(1, 6):
    # YOUR CODE HERE
    # 1. Call the API with a system prompt that asks the model to think step by step,
    #    track the water level at each phase transition,
    #    then give the final answer on the last line as just the number
    # 2. Extract the final number from the response
    # 3. Compare to TANK_ANSWER
    answer = ""  # replace
    reasoning = ""  # replace — the model's step-by-step reasoning
    print(f"  Run {run}: {answer}  (reasoning: {reasoning[:80]}...)")

print(f"\nCoT: {cot_correct}/5 correct")
print(f"Direct was: {direct_correct}/5 correct")

---
## Exercise 4: Extract → Verify Loop

Extraction is never perfect on the first pass. A simple but powerful pattern:

1. **Extract** — ask the model to pull out structured data (e.g., place names)
2. **Verify** — send the extracted data *and* the original text back to the model, asking: "Did I miss anything? Did I include anything wrong?"

This two-pass approach catches mistakes that a single prompt misses — hallucinated entities, overlooked mentions, ambiguous references.

You'll extract **place names** from `sherlock_text`, then verify the result with a second prompt.

In [ ]:
# Part A: Extract place names from sherlock_text.
#
# Write a prompt that asks the model to find all place names (cities, streets,
# countries, regions) mentioned in the text. Return them as a JSON list of strings.
#
# Use WEAK_MODEL to make the extraction imperfect — that's the point!

# YOUR CODE HERE
# 1. Write a system prompt asking for place names as a JSON list
# 2. Call client.chat.completions.create(...) with WEAK_MODEL
# 3. Parse the result

places = None  # replace with your parsed list

print("=== Extracted places ===")
print(json.dumps(places, indent=2))
print(f"\nFound {len(places)} places. But are they all correct? Did we miss any?")

In [ ]:
# Part B: Verify the extraction with a second prompt.
#
# Send BOTH the original text AND your extracted places to the model (use MODEL,
# not WEAK_MODEL). Ask it to check:
#   - Are there any places in the text that were missed?
#   - Are there any items in the list that are NOT actually places in the text?
#   - Return the corrected list as JSON
#
# Compare the two lists — what changed?

# YOUR CODE HERE
# 1. Write a verification prompt that includes sherlock_text and your places list
# 2. Call client.chat.completions.create(...) with MODEL
# 3. Parse and compare

verified_places = None  # replace with your parsed list

print("=== Verified places ===")
print(json.dumps(verified_places, indent=2))
print(f"\nBefore: {len(places)} places")
print(f"After:  {len(verified_places)} places")
print(f"\nAdded:   {set(verified_places) - set(places) if places and verified_places else '?'}")
print(f"Removed: {set(places) - set(verified_places) if places and verified_places else '?'}")

In [ ]:
# Part C (bonus): Try the same extract → verify pattern with a different category.
#
# Instead of places, try extracting: person names, dates, or cases mentioned.
# Does the verify step catch different kinds of errors for different categories?

# YOUR CODE HERE

---
## Exercise 5: Multi-Document Extraction with Deduplication

Real extraction pipelines process **multiple documents** about the same topic, then merge and deduplicate the results. "Tim Cook" and "Cook" are the same person. "MSFT" and "Microsoft" are the same org.

Below are three short texts about the same event. Your task:

1. Extract entities from each text separately
2. Merge all entities into one list
3. Deduplicate: group entities that refer to the same real-world thing
4. For deduplication, you can use the LLM itself or write heuristic code (or both!)

In [ ]:
# Three texts about Microsoft's acquisition of GitHub

ARTICLES = {
    "article_1": (
        "Microsoft announced on Monday that it would acquire GitHub, the world's "
        "largest code-hosting platform, for $7.5 billion in stock. CEO Satya Nadella "
        "called it a strategic move to strengthen Microsoft's developer tools. "
        "The deal is expected to close by the end of 2018."
    ),
    "article_2": (
        "MSFT is buying GitHub for $7.5B — the biggest acquisition since LinkedIn. "
        "Nadella said the deal will let developers 'do their best work.' "
        "GitHub CEO Chris Wanstrath will stay on as a technical fellow. "
        "The EU still needs to approve the acquisition."
    ),
    "article_3": (
        "Developers on Hacker News are divided over the Microsoft-GitHub deal. "
        "Some worry that MS will ruin the platform like it did with Skype. "
        "Others point out that Nat Friedman, the new GitHub CEO, is a respected "
        "open-source advocate from Xamarin. The $7.5 billion price tag raised eyebrows "
        "in San Francisco's tech community."
    ),
}

In [ ]:
# Step 1: Extract entities from each article.

all_entities = []  # list of (source_article, entity_dict) tuples

for name, text in ARTICLES.items():
    raw, parsed = extract_entities(text, model=MODEL, temperature=0)
    if parsed:
        for e in parsed["entities"]:
            all_entities.append((name, e))
        print(f"{name}: {len(parsed['entities'])} entities")
    else:
        print(f"{name}: ✗ extraction failed")

print(f"\nTotal raw entities: {len(all_entities)}")
for source, e in all_entities:
    print(f"  [{source}] {e['text']:30s}  {e['type']}")

In [ ]:
# Step 2: Deduplicate entities.
#
# Approach A (LLM-based): send the full entity list to the model and ask it to
#   group entities that refer to the same real-world thing. Return a list of
#   {"canonical": "Microsoft", "type": "ORGANIZATION", "aliases": ["MSFT", "MS"], "sources": [...]}
#
# Approach B (heuristic): use string similarity, substring matching, or abbreviation
#   expansion to cluster entities.
#
# You can use either approach or combine them.

def deduplicate_entities(entities: list[tuple[str, dict]]) -> list[dict]:
    """
    Group entities that refer to the same real-world thing.

    Args:
        entities: list of (source_name, entity_dict) tuples
                  where entity_dict has keys: text, type

    Returns:
        list of {
            "canonical": str,        # preferred name
            "type": str,             # entity type
            "aliases": list[str],    # other names found
            "sources": list[str],    # which articles mentioned it
        }
    """
    # YOUR CODE HERE
    pass


deduped = deduplicate_entities(all_entities)

print(f"\n=== Deduplicated: {len(deduped)} unique entities ===\n")
for group in deduped:
    aliases = ", ".join(group["aliases"]) if group["aliases"] else "(none)"
    sources = ", ".join(group["sources"])
    print(f"  {group['canonical']:25s}  [{group['type']}]")
    print(f"    Aliases: {aliases}")
    print(f"    Sources: {sources}\n")

---
## Summary

You've now practiced the core prompting techniques and built up to production-grade extraction:

| Exercise | Skill |
|---|---|
| 0. Warm-Up | Loading text, writing a prompt, parsing JSON — the basic loop |
| 1. Schema Design | Designing the contract before writing the prompt |
| 2. Few-Shot Learning | Using examples to steer output style and vocabulary |
| 3. Chain-of-Thought | Step-by-step reasoning for hard problems |
| 4. Extract → Verify | Two-pass extraction that catches its own mistakes |
| 5. Multi-doc + Dedup | End-to-end extraction pipeline across sources |

**Next up:** cost-aware extraction strategies (chunking vs. whole-document) and LLM-as-Judge evaluation.